# Testing Stages 1 & 2: Scoring and Ingest

This notebook walks through the two implemented pipeline stages step by step:

1. **Stage 1 — Scoring**: Takes raw earnings call transcripts and produces structured sentiment scores using two methods (LLM-based and dictionary-based).
2. **Stage 2 — Ingest**: Retrieves earnings call transcripts from the Nuvolos/Snowflake database.

We will test each component individually, explain every concept, and show how the pieces fit together.

---

## Key definitions

| Term | Definition |
|------|-----------|
| **Earnings call** | A quarterly conference call where a public company's management discusses financial results with analysts and investors. The transcript is the raw text record of this call. |
| **Sentiment scoring** | The process of assigning a positive, negative, or neutral label to statements extracted from text. In our pipeline, this is done per financial category. |
| **Forward-looking statement** | A sentence that discusses future expectations, guidance, or projections (as opposed to reporting past results). These are the most valuable for prediction. |
| **LLM scoring** | Using a Large Language Model (e.g. GPT-5.4-nano) to extract, classify, and score sentences from transcripts through a multi-stage prompt pipeline. |
| **Loughran-McDonald (LM) scoring** | A deterministic, dictionary-based method that counts words belonging to predefined sentiment categories (Positive, Negative, Uncertainty, etc.) from a financial lexicon. |
| **Information Coefficient (IC)** | A correlation measure between predicted scores and realised outcomes. Higher IC = better predictive power. |
| **Out-of-sample R-squared (OOS R²)** | The proportion of variance in the target variable explained by the model on data it was *not* trained on. This is our primary optimisation metric. |

---

# Part 1: Global Settings

The `Settings` object is the single source of truth for all pipeline configuration.
It loads values from `configs/global_params.yaml` and can be overridden by environment variables.

Every pipeline stage receives a `Settings` instance — no module reads config files or environment variables directly.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

from pipeline.settings import load_settings

settings = load_settings()

print("=== Global Settings (loaded from configs/global_params.yaml) ===\n")
print(f"  LLM model:             {settings.llm_model}")
print(f"  LLM base URL:          {settings.llm_base_url}")
print(f"  LLM temperature:       {settings.llm_temperature}")
print(f"  LLM max workers:       {settings.llm_max_workers}")
print(f"  LLM structured output: {settings.llm_supports_structured_output}")
print(f"  LLM think mode:        {settings.llm_think_mode}")
print(f"  LLM max tokens:        {settings.llm_max_tokens}")
print(f"  OpenAI API key set:    {'yes' if settings.openai_api_key else 'NO (set OPENAI_API_KEY env var)'}")
print()
print(f"  Chunk size:            {settings.chunk_size} sentences")
print(f"  Chunk overlap:         {settings.chunk_overlap} sentences")
print(f"  Neutering enabled:     {settings.neutering_enabled}")
print()
print(f"  Target variable:       {settings.target_variable}")
print(f"  Target direction:      {settings.target_direction}")
print(f"  Eval metric:           {settings.eval_metric}")
print(f"  Eval horizon:          {settings.eval_horizon_days} days")
print(f"  Random seed:           {settings.random_seed}")
print()
print(f"  Feature set:           {settings.feature_set}")
print(f"  Winsorize quantiles:   {settings.winsorize_quantiles}")
print(f"  Train/test split year: {settings.train_test_split_year}")
print(f"  Exclude FYEAR >=:      {settings.exclude_fyear_gte}")
print(f"  Universe filter:       {settings.universe_filter}")
print(f"  Min history quarters:  {settings.min_history_quarters}")

### Settings validation

The `Settings` class validates all inputs via Pydantic. For example:
- The `scoring_prompt` **must** contain `{context}` and `{target}` placeholders
- `winsorize_quantiles` must be a 2-element list where `0 < lower < upper < 1`
- `llm_temperature` must be between 0.0 and 2.0
- `eval_metric` must be one of: `oos_r_squared`, `ic`, `icir`, `hit_rate`, `sharpe`

Let's verify validation catches bad inputs:

In [ ]:
from pydantic import ValidationError
from pipeline.settings import Settings

# Test 1: Missing {context} placeholder in prompt
try:
    Settings(scoring_prompt="No placeholders for {target} here.")
    print("FAIL: Should have raised ValidationError")
except ValidationError as e:
    print(f"OK: Caught invalid prompt — {e.errors()[0]['msg']}")

# Test 2: Reversed winsorize quantiles
try:
    Settings(winsorize_quantiles=[0.99, 0.01])
    print("FAIL: Should have raised ValidationError")
except ValidationError as e:
    print(f"OK: Caught invalid quantiles — {e.errors()[0]['msg']}")

# Test 3: Temperature out of range
try:
    Settings(llm_temperature=5.0)
    print("FAIL: Should have raised ValidationError")
except ValidationError as e:
    print(f"OK: Caught invalid temperature — {e.errors()[0]['msg']}")

# Test 4: Valid settings pass
s = Settings(scoring_prompt="Evaluate {context} for {target}.", llm_temperature=0.5)
print(f"\nOK: Valid settings created with temperature={s.llm_temperature}")

---

# Part 2: Stage 1 — The Scoring Pipeline

The scoring pipeline transforms raw earnings call text into structured sentiment scores. There are **two independent scoring methods**:

## Method A: LLM-based scoring (5-stage pipeline)

This is the core innovation of the project. A Large Language Model processes each transcript through 5 sequential stages:

```
Raw transcript
    |
    v
[1. EXTRACTION]     Extract forward-looking financial statements
    |
    v
[2. DEDUPLICATION]  Remove redundant sentences
    |
    v
[3. FILTRATION]     Keep only FUTURE-oriented sentences (discard PAST)
    |
    v
[4. CATEGORIZATION] Assign each sentence to 1 of 7 financial categories
    |
    v
[5. SENTIMENT]      Label each sentence as positive / negative / neutral
    |
    v
Per-category scores (linear_score, log_score)
```

## Method B: Loughran-McDonald dictionary scoring

A deterministic method (no LLM needed) that counts words from a financial sentiment dictionary. Much faster but less nuanced than the LLM approach.

---

Let's test each component individually.

## 2.1 The seven financial categories

Every sentence extracted from an earnings call is classified into exactly **one** of these categories. They represent the key dimensions of a company's financial outlook:

| Category | What it covers |
|----------|---------------|
| **REVENUE** | Sales, turnover, top-line growth, pricing, market share |
| **INDUSTRY/MOATS/DRIVERS** | Competitive advantages, market dynamics, industry trends |
| **EARNING & COSTS** | Margins, EBITDA, operating costs, cost reduction, profitability |
| **CAP ALLOCATION/CASH** | CapEx, M&A, dividends, buybacks, debt management, R&D |
| **EXOGENOUS** | Macro factors (rates, FX, inflation), regulation, geopolitics |
| **MANAGEMENT/CULTURE/SUSTAINABILITY** | Leadership quality, ESG, corporate governance |
| **OTHER CRITERIA** | Anything that doesn't fit the above |

Each category gets a **sentiment breakdown** (positive / negative / neutral counts) and two scores:
- **Linear score**: `(positive - negative) / (positive + negative)` — ranges from -1 to +1
- **Log score**: `log((pos+1)/(neg+1)) / log(pos+neg+2)` — dampened version, less sensitive to extremes

In [ ]:
from pipeline.scoring.constants import CATEGORIES, SENTIMENTS

print("Financial categories used in the pipeline:")
for i, cat in enumerate(CATEGORIES, 1):
    print(f"  {i}. {cat}")

print(f"\nSentiment labels: {SENTIMENTS}")
print(f"\nTotal combinations: {len(CATEGORIES)} categories x {len(SENTIMENTS)} sentiments = {len(CATEGORIES) * len(SENTIMENTS)} dimensions")

## 2.2 Transcript chunking

Earnings call transcripts are typically 30,000-40,000 characters long. LLMs have context limits and work better with focused input, so we split transcripts into **chunks** of N sentences each.

**Key parameters** (from `global_params.yaml`):
- `chunk_size`: Number of sentences per chunk (default: 30)
- `chunk_overlap`: How many sentences overlap between consecutive chunks (default: 0)

**How it works**: The text is split on `. ` (period + space), then grouped into chunks. Chunks shorter than 50 characters are discarded.

Let's test with a synthetic transcript:

In [ ]:
from pipeline.scoring.chunker import chunk_transcript

# Create a synthetic transcript with 90 sentences
synthetic_transcript = ". ".join(
    f"Sentence {i} discusses the company's financial outlook and strategic priorities"
    for i in range(1, 91)
)

print(f"Transcript length: {len(synthetic_transcript):,} characters")
print(f"Approximate sentences: 90\n")

# Chunk with default settings (30 sentences, no overlap)
chunks_no_overlap = chunk_transcript(synthetic_transcript, chunk_size=30, overlap=0)
print(f"Chunks (size=30, overlap=0): {len(chunks_no_overlap)}")
for i, c in enumerate(chunks_no_overlap):
    print(f"  Chunk {i+1}: {len(c):,} chars, starts with: '{c[:60]}...'")

# Chunk with overlap
print()
chunks_with_overlap = chunk_transcript(synthetic_transcript, chunk_size=30, overlap=10)
print(f"Chunks (size=30, overlap=10): {len(chunks_with_overlap)}")
print(f"  -> More chunks because sentences are shared between consecutive chunks")

# Edge cases
print(f"\nEmpty text:  {len(chunk_transcript('', chunk_size=30))} chunks")
print(f"Short text:  {len(chunk_transcript('Too short.', chunk_size=30))} chunks (filtered < 50 chars)")

## 2.3 Entity neutering (optional robustness step)

**Neutering** replaces named entities (people, companies, dates) in text with placeholder tokens like `[PERSON]`, `[ORG]`, `[DATE]`. This tests whether the LLM's scoring is influenced by *who* said something rather than *what* was said.

For example:
- `"Tim Cook announced Apple will grow revenue by 15%"` becomes `"[PERSON] announced [ORG] will grow revenue by 15%"`

This helps measure **robustness**: if scores change significantly after neutering, the model may be relying on entity recognition rather than financial content.

**Note**: Neutering requires `spacy` and the `en_core_web_sm` model. We'll test the helper functions that don't require spaCy:

In [ ]:
from pipeline.scoring.neutering import _apply_replacements, _drop_duplicated_tokens

# Test entity replacement
text = "John Smith met Jane Doe at Apple headquarters on Monday"
spans = [
    (0, 10, "[PERSON]"),   # "John Smith"
    (15, 23, "[PERSON]"),  # "Jane Doe"
    (27, 32, "[ORG]"),     # "Apple"
    (50, 56, "[DATE]"),    # "Monday"
]

neutered = _apply_replacements(text, spans)
print(f"Original:  {text}")
print(f"Neutered:  {neutered}")

# Test deduplication of consecutive tokens
text_with_dupes = "[PERSON] [PERSON] said [ORG], [ORG] will grow"
cleaned = _drop_duplicated_tokens(text_with_dupes)
print(f"\nBefore dedup: {text_with_dupes}")
print(f"After dedup:  {cleaned}")
print(f"  -> Consecutive duplicate tokens are collapsed into one")

## 2.4 Pydantic schemas for structured LLM output

Each of the 5 pipeline stages expects the LLM to return **structured JSON** matching a specific Pydantic schema. This ensures:
1. The LLM's output is machine-parseable (not free-form text)
2. Missing or invalid fields are caught immediately
3. Downstream stages always receive well-typed data

Let's examine what each stage expects:

In [ ]:
import json
from pipeline.scoring.schemas import (
    KeySentencesWithReason,
    KeptSentences,
    FutureStatement,
    CategorizedSentence,
    SentimentSentence,
    EarningsCallResult,
    CategoryScore,
)

stages = [
    ("Stage 1: Extraction",     KeySentencesWithReason),
    ("Stage 2: Deduplication",  KeptSentences),
    ("Stage 3: Filtration",     FutureStatement),
    ("Stage 4: Categorization", CategorizedSentence),
    ("Stage 5: Sentiment",      SentimentSentence),
]

for name, schema in stages:
    print(f"=== {name} ===")
    print(f"  Schema: {schema.__name__}")
    schema_dict = schema.model_json_schema()
    # Show required fields
    props = schema_dict.get("properties", {})
    for field, info in props.items():
        field_type = info.get("type", info.get("$ref", "complex"))
        print(f"  - {field}: {field_type}")
    print()

# Show the final composite result
print("=== Final Output: EarningsCallResult ===")
print("  Fields: company_name, year, quarter, company_id, transcript_id, model_used")
print("  scores: dict[category -> CategoryScore(positive, negative, neutral, linear_score, log_score)]")
print("  sentences: list[ScoredSentence(text, category, sentiment, explanation)]")

## 2.5 Score computation

After the 5 LLM stages produce a list of scored sentences, the **score computer** aggregates them into per-category metrics.

### Formulas

For each of the 7 categories, given `p` positive and `n` negative sentences:

**Linear score** = `(p - n) / (p + n)` if `p + n > 0`, else `None`
- Range: [-1, +1]
- Simple ratio: +1 means all positive, -1 means all negative

**Log score** = `log((p + 1) / (n + 1)) / log(p + n + 2)` if `p + n > 0`, else `None`
- Range: approximately [-1, +1]
- The +1 smoothing (Laplace) prevents division by zero
- The denominator normalises by total count, making the score comparable across categories with different sentence volumes

Let's compute scores on synthetic data:

In [ ]:
from pipeline.scoring.score_computer import compute_scores, build_result

# Simulate scored sentences (as if the LLM pipeline produced them)
synthetic_sentences = [
    # Revenue: mostly positive
    {"category": "REVENUE", "sentiment": "positive"},
    {"category": "REVENUE", "sentiment": "positive"},
    {"category": "REVENUE", "sentiment": "positive"},
    {"category": "REVENUE", "sentiment": "negative"},
    {"category": "REVENUE", "sentiment": "neutral"},
    # Earnings & Costs: mixed
    {"category": "EARNING & COSTS", "sentiment": "positive"},
    {"category": "EARNING & COSTS", "sentiment": "negative"},
    {"category": "EARNING & COSTS", "sentiment": "negative"},
    {"category": "EARNING & COSTS", "sentiment": "neutral"},
    # Exogenous: negative (macro headwinds)
    {"category": "EXOGENOUS", "sentiment": "negative"},
    {"category": "EXOGENOUS", "sentiment": "negative"},
    {"category": "EXOGENOUS", "sentiment": "neutral"},
]

scores = compute_scores(synthetic_sentences)

print(f"{'Category':<40} {'Pos':>4} {'Neg':>4} {'Neu':>4} {'Linear':>8} {'Log':>8}")
print("=" * 72)
for cat, sc in scores.items():
    lin = f"{sc.linear_score:+.3f}" if sc.linear_score is not None else "   N/A"
    log = f"{sc.log_score:+.3f}" if sc.log_score is not None else "   N/A"
    print(f"{cat:<40} {sc.positive:>4} {sc.negative:>4} {sc.neutral:>4} {lin:>8} {log:>8}")

print("\nInterpretation:")
print("  REVENUE:        +0.500 linear -> 3 pos vs 1 neg -> bullish revenue outlook")
print("  EARNING & COSTS: -0.333 linear -> 1 pos vs 2 neg -> cost pressures")
print("  EXOGENOUS:      -1.000 linear -> 0 pos vs 2 neg -> negative macro environment")
print("  Others:           N/A -> no sentences in these categories")

## 2.6 Building a complete EarningsCallResult

The `build_result()` function combines scored sentences with metadata to produce the final output object. This is what gets serialised to JSON and stored per company-quarter.

In [ ]:
# Build a full result object with metadata
sentences_with_text = [
    {"text": "We expect revenue to grow 15% next quarter.",
     "category": "REVENUE", "sentiment": "positive", "reason_sentiment": "Growth projection"},
    {"text": "Rising input costs will compress margins by 200bps.",
     "category": "EARNING & COSTS", "sentiment": "negative", "reason_sentiment": "Cost pressure"},
    {"text": "We plan to increase CapEx by $2B for AI infrastructure.",
     "category": "CAP ALLOCATION/CASH", "sentiment": "positive", "reason_sentiment": "Strategic investment"},
]

result = build_result(
    company_name="Acme Corp",
    year="2025",
    quarter="4",
    company_id="12345",
    transcript_id="67890",
    model_used=settings.llm_model,
    sentences=sentences_with_text,
)

print(f"Company: {result.company_name} — {result.year} Q{result.quarter}")
print(f"Model:   {result.model_used}")
print(f"Sentences scored: {len(result.sentences)}")
print()

# Show the JSON structure (what gets saved to disk)
result_dict = result.model_dump()
print("JSON structure (truncated):")
print(json.dumps(result_dict, indent=2, default=str)[:1200])
print("...")

## 2.7 The LLM client

The `LLMClient` is the centralised interface for all LLM interactions. Key design decisions:

- **Provider-agnostic**: Uses the OpenAI SDK which is compatible with OpenAI, ollama, LM Studio, vLLM, DeepSeek, and other providers
- **Structured output**: When the provider supports it (`llm_supports_structured_output=True`), sends a JSON schema so the LLM returns valid JSON. Otherwise, injects the schema into the prompt as an instruction.
- **Think mode**: When `llm_think_mode=True`, prepends `/think` to the user prompt, triggering chain-of-thought reasoning in supported models
- **Markdown fence stripping**: Many LLMs wrap JSON in ` ```json ... ``` ` — the client strips these automatically
- **Token tracking**: Every call accumulates input/output token counts for cost monitoring

Let's test with a mock (no real API call needed):

In [ ]:
from unittest.mock import MagicMock, patch
from pipeline.llm import LLMClient, TokenUsage
from pipeline.scoring.schemas import KeySentencesWithReason

# Create a client with test settings (no real API call)
test_settings = Settings(
    openai_api_key="test-key",
    llm_model="gpt-5.4-nano",
    llm_base_url="http://localhost:1234/v1",
    llm_supports_structured_output=False,
)
client = LLMClient(test_settings)

# Mock the OpenAI API response
mock_response = MagicMock()
mock_response.choices = [MagicMock(message=MagicMock(
    content=json.dumps({
        "sentences": [
            {"sentence": "We expect revenue to grow 15% next quarter.", "reason": "Forward-looking with quantifiable projection."},
            {"sentence": "Margins will expand by 200bps through cost optimisation.", "reason": "Clear financial KPI impact."},
        ]
    })
))]
mock_response.usage = MagicMock(prompt_tokens=150, completion_tokens=80)

# Call the client with mocked API
with patch.object(client._client.chat.completions, "create", return_value=mock_response):
    parsed, tok_in, tok_out = client.complete(
        system_prompt="You are a financial analyst.",
        user_prompt="Extract forward-looking statements from this transcript chunk.",
        schema=KeySentencesWithReason,
    )

print("Parsed LLM response:")
for i, s in enumerate(parsed["sentences"], 1):
    print(f"  {i}. \"{s['sentence']}\"")
    print(f"     Reason: {s['reason']}")

print(f"\nToken usage: {tok_in} input + {tok_out} output = {tok_in + tok_out} total")

# Test markdown fence stripping
mock_response2 = MagicMock()
mock_response2.choices = [MagicMock(message=MagicMock(
    content='```json\n{"sentences": [{"sentence": "Test", "reason": "R"}]}\n```'
))]
mock_response2.usage = MagicMock(prompt_tokens=10, completion_tokens=5)

with patch.object(client._client.chat.completions, "create", return_value=mock_response2):
    parsed2, _, _ = client.complete("sys", "user", schema=KeySentencesWithReason)

print(f"\nMarkdown fence stripping: '{parsed2['sentences'][0]['sentence']}' (extracted from ```json block)")

## 2.8 Full LLM pipeline (mocked end-to-end)

Now let's run the complete 5-stage pipeline on a synthetic transcript, with all LLM calls mocked.

This demonstrates the full flow: **chunking -> extraction -> deduplication -> filtration -> categorization -> sentiment -> scoring**.

In [ ]:
from typing import Any
from pipeline.scoring.llm_pipeline import LLMScoringPipeline

def make_mock_client():
    """Build a mock LLM client that returns realistic responses for each pipeline stage."""
    mock = MagicMock()

    def side_effect(system_prompt: str, user_prompt: str, schema: Any = None):
        if schema and schema.__name__ == "KeySentencesWithReason":
            return ({"sentences": [
                {"sentence": "We expect revenue to grow 15% next quarter.", "reason": "Forward-looking with quantifiable projection."},
                {"sentence": "Operating margins will expand by 200bps.", "reason": "Clear causal impact on financial KPI."},
            ]}, 100, 50)
        elif schema and schema.__name__ == "KeptSentences":
            return ({"sentences": [
                {"number": 0, "text": "We expect revenue to grow 15% next quarter."},
                {"number": 1, "text": "Operating margins will expand by 200bps."},
            ]}, 80, 30)
        elif schema and schema.__name__ == "FutureStatement":
            text = "We expect revenue to grow 15% next quarter."
            if "margins" in user_prompt.lower():
                text = "Operating margins will expand by 200bps."
            return ({"text": text, "classification": "FUTURE", "confidence": "HIGH",
                     "explanation": "Forward-looking projection."}, 60, 20)
        elif schema and schema.__name__ == "CategorizedSentence":
            cat = "REVENUE" if "revenue" in user_prompt.lower() else "EARNING & COSTS"
            return ({"text": user_prompt[:50], "category": cat,
                     "reason": f"Classified as {cat}."}, 60, 20)
        elif schema and schema.__name__ == "SentimentSentence":
            return ({"text": user_prompt[:50], "sentiment": "positive",
                     "reason": "Growth and expansion are positive."}, 60, 20)
        return ({}, 0, 0)

    mock.complete = MagicMock(side_effect=side_effect)
    return mock


# Create pipeline with test settings
pipeline_settings = Settings(
    openai_api_key="test",
    llm_model="gpt-5.4-nano",
    llm_base_url="http://localhost:1234/v1",
    chunk_size=30,
)
pipeline = LLMScoringPipeline(pipeline_settings)
pipeline.client = make_mock_client()

# Create a synthetic transcript (60 sentences)
transcript = ". ".join(
    f"Sentence {i} about the company financial outlook and strategic growth initiatives"
    for i in range(60)
)

print(f"Input transcript: {len(transcript):,} chars, ~60 sentences\n")
print("Running 5-stage LLM pipeline...")

result = pipeline.run(
    transcript,
    company_name="Acme Corp",
    year="2025",
    quarter="4",
    company_id="12345",
    transcript_id="67890",
)

print(f"\n{'='*60}")
print(f"Result: {result['company_name']} — {result['year']} Q{result['quarter']}")
print(f"Model: {result['model_used']}")
print(f"Sentences scored: {len(result['sentences'])}")
print(f"{'='*60}")

print(f"\n{'Category':<40} {'Pos':>4} {'Neg':>4} {'Neu':>4} {'Linear':>8}")
print("-" * 65)
for cat, sc in result["scores"].items():
    if sc["positive"] + sc["negative"] + sc["neutral"] > 0:
        lin = f"{sc['linear_score']:+.3f}" if sc["linear_score"] is not None else "   N/A"
        print(f"{cat:<40} {sc['positive']:>4} {sc['negative']:>4} {sc['neutral']:>4} {lin:>8}")

print(f"\nToken usage: {pipeline.usage.input_tokens:,} input + {pipeline.usage.output_tokens:,} output")

## 2.9 Method B: Loughran-McDonald dictionary scoring

The **Loughran-McDonald (LM)** financial dictionary is an alternative to LLM scoring. It classifies individual words (not sentences) into 7 sentiment categories:

| LM Category | Definition | Example words |
|-------------|-----------|---------------|
| **Positive** | Words conveying favourable outlook | good, gain, improve, profit, strong |
| **Negative** | Words conveying adverse outlook | loss, decline, risk, impair, weak |
| **Uncertainty** | Words indicating ambiguity | may, possible, uncertain, approximate |
| **Litigious** | Legal and regulatory language | court, lawsuit, regulation, compliance |
| **Strong_Modal** | Words of strong obligation/intent | must, shall, will, always |
| **Weak_Modal** | Words of weak possibility | could, might, possibly, somewhat |
| **Constraining** | Words about limitations | restrict, limit, prohibit, obligation |

**How it works:**
1. Tokenise the transcript (uppercase, strip punctuation)
2. Look up each word in the dictionary
3. Count how many words fall into each category
4. Compute linear and log scores from Positive vs Negative counts

**Advantages**: Deterministic, fast, reproducible, no API cost.
**Disadvantages**: No context awareness (e.g. "not good" counts "good" as positive), no sentence-level granularity.

Let's test with a small dictionary:

In [ ]:
import os
import tempfile
import pandas as pd
from pipeline.scoring.lm_pipeline import LMScoringPipeline, _tokenize, load_lm_dictionary

# Step 1: Understand tokenisation
sample_text = "Revenue grew 15% year-over-year, driven by strong demand!"
tokens = _tokenize(sample_text)
print(f"Original: {sample_text}")
print(f"Tokens:   {tokens}")
print(f"  -> Uppercased, non-alpha stripped, split on whitespace\n")

# Step 2: Create a small test dictionary
dict_data = pd.DataFrame({
    "Word":        ["STRONG", "GROWTH", "GREW", "DEMAND", "RISK", "DECLINE", "WEAK", "LOSS",
                    "UNCERTAIN", "POSSIBLY", "MUST", "COULD", "RESTRICT"],
    "Positive":    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "Negative":    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0],
    "Uncertainty": [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    "Litigious":   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "Strong_Modal":[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    "Weak_Modal":  [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    "Constraining":[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
})

# Save to temp file and load
with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as f:
    dict_data.to_csv(f, index=False)
    dict_path = f.name

word_map = load_lm_dictionary(dict_path)
print(f"Dictionary loaded: {len(word_map)} categorised words")
for word, cats in sorted(word_map.items()):
    print(f"  {word:<12} -> {cats}")

# Step 3: Score a sample transcript
sample_transcript = """
Revenue grew strongly in Q4, driven by robust demand across all segments.
However, we face risk of decline in emerging markets due to uncertain macro conditions.
Management must restrict discretionary spending, though growth could possibly continue.
The weak consumer environment and potential loss of market share remain concerns.
"""

lm_pipeline = LMScoringPipeline(dict_path)
lm_result = lm_pipeline.run(
    sample_transcript,
    company_name="TestCo",
    year="2025",
    quarter="4",
)

print(f"\n{'='*50}")
print(f"LM Scoring Result: {lm_result['company_name']} {lm_result['year']} Q{lm_result['quarter']}")
print(f"{'='*50}")
print(f"Total words analysed: {lm_result['total_words']}")
print(f"\nSentiment counts:")
for cat, count in lm_result["sentiment_counts"].items():
    if count > 0:
        print(f"  {cat:<15}: {count}")
print(f"\nLinear score: {lm_result['linear_score']:.4f}")
print(f"Log score:    {lm_result['log_score']:.4f}")
print(f"\nInterpretation: linear={lm_result['linear_score']:.3f} means")
if lm_result['linear_score'] and lm_result['linear_score'] > 0:
    print(f"  More positive than negative words -> mildly positive tone")
elif lm_result['linear_score'] and lm_result['linear_score'] < 0:
    print(f"  More negative than positive words -> negative tone")

os.unlink(dict_path)

---

# Part 3: Stage 2 — Data Ingest

The ingest stage retrieves raw earnings call transcripts from the **Nuvolos** database (built on Snowflake). This is the data source for the entire pipeline.

## 3.1 How transcript retrieval works

The data lives in Snowflake tables provided by S&P Capital IQ via the Nuvolos academic platform:

```
ciqTranscript           — transcript metadata (creation date, type)
ciqTranscriptComponent  — individual text blocks (paragraphs/speakers)
ciqEventPit             — event information (headline, key dev ID)
ciqCompany              — company metadata
ciqEventType            — event type (we filter for type 48 = Earnings Calls)
ciqEventCallBasicInfo   — fiscal quarter/year info
```

These 6+ tables are joined via a SQL query, then the transcript is **reconstructed** by:

1. **Priority filtering**: Audited transcripts (type 8) > Edited (2) > Proofed (1)
2. **Section filtering**: Keep only Main presentation (type 2) and Q&A (type 4)
3. **Deduplication**: Remove duplicate text blocks
4. **Ordering**: Sort by `componentOrder` to reconstruct the original speech flow

The result is a single `EarningsCall` dataclass with:
- `company_name`, `year`, `quarter` — identifiers
- `company_id`, `transcript_id` — database IDs
- `transcript` — the full reconstructed text (typically 30,000-40,000 characters)

## 3.2 Authentication methods

The `connect_nuvolos()` function tries three authentication methods in order:

| Method | When it works | How to configure |
|--------|--------------|-----------------|
| **Token auth** | Inside Nuvolos or with allowlisted IP | Set `NUVOLOS_SF_TOKEN` env var |
| **RSA key pair** | From any IP with registered key | Set `SNOWFLAKE_RSA_KEY` env var pointing to private key file |
| **Browser SSO** | From any IP (fallback) | Opens browser for Okta/SAML 2FA login |

**Environment variables** (set these before connecting):
```bash
export NUVOLOS_USERNAME="your.email@school.edu"
export NUVOLOS_SF_TOKEN="your_token_here"     # optional, for token auth
export SNOWFLAKE_RSA_KEY="/path/to/rsa_key"   # optional, for key auth
```

In [ ]:
from pipeline.ingest.transcript import (
    EarningsCall,
    _build_query,
    _reconstruct_transcript,
)

# Inspect the EarningsCall dataclass
print("EarningsCall dataclass fields:")
for field_name in EarningsCall.__dataclass_fields__:
    print(f"  - {field_name}")

# Show the SQL query that would be generated (without executing it)
print("\n" + "="*60)
print("Sample SQL query for Apple Inc. 2024 Q3:")
print("="*60)
query = _build_query(
    company_name="Apple Inc.",
    year="2024",
    quarter="3",
    db_name="essec_metalab/ako_earnings_prediction",
    schema_name="master/development",
)
# Show just the key parts
print(query[:800])
print("...")

## 3.3 Transcript reconstruction logic

The raw Snowflake data contains multiple versions of the same transcript (audited, edited, proofed) and multiple sections (presentation, Q&A, operator). The reconstruction logic selects the **best version** and **relevant sections**.

Let's test this with synthetic data that mimics the Snowflake output:

In [ ]:
# Simulate raw Snowflake output with multiple transcript versions and sections
raw_data = pd.DataFrame({
    "transcriptcollectiontypeid": [
        2, 2, 2, 2, 2,    # Edited version (priority 2)
        8, 8, 8, 8, 8,    # Audited version (priority 8, best)
    ],
    "documentobjectreltypeid": [None]*10,
    "transcriptcreationdateutc": ["2024-10-30"] * 10,
    "componenttext": [
        # Edited version
        "Good morning everyone.",                              # Operator intro
        "Thank you. Revenue grew 12% this quarter.",           # Presentation
        "We expect continued momentum into Q1.",               # Presentation
        "Let me open it up for questions.",                    # Q&A transition
        "Can you discuss margin trends?",                      # Q&A
        # Audited version (same content, higher quality)
        "Good morning and welcome to the call.",               # Operator (type 1)
        "Revenue grew 12% year-over-year to $45B.",           # Main (type 2)
        "We project 15% growth for the next quarter.",        # Main (type 2)
        "Turning to Q&A, first question please.",              # Q&A (type 4)
        "What is your outlook on margins?",                    # Q&A (type 4)
    ],
    "transcriptcomponenttypeid": [
        1, 2, 2, 4, 4,    # Edited: 1=operator, 2=main, 4=Q&A
        1, 2, 2, 4, 4,    # Audited: same structure
    ],
    "componentorder": [1, 2, 3, 4, 5, 1, 2, 3, 4, 5],
    "companyid": [12345] * 10,
    "transcriptid": [67890] * 10,
})

# Run the reconstruction
reconstructed = _reconstruct_transcript(raw_data.copy())

print("Reconstruction logic:")
print(f"  Input rows:  {len(raw_data)}")
print(f"  Output rows: {len(reconstructed)}")
print(f"  Selected version: Audited (priority 8)")
print(f"  Kept sections: Main (type 2) + Q&A (type 4)")
print(f"  Filtered out: Operator intro (type 1)")
print()
print("Reconstructed transcript:")
for _, row in reconstructed.iterrows():
    section = "MAIN" if row["transcriptcomponenttypeid"] == 2 else "Q&A"
    print(f"  [{section}] {row['componenttext']}")

## 3.4 Live connection (optional)

**Uncomment and run the cell below to connect to Nuvolos and retrieve real transcripts.** This requires:
1. A valid Nuvolos account
2. Environment variables set (`NUVOLOS_USERNAME`, optionally `NUVOLOS_SF_TOKEN`)
3. Network access to Snowflake

If you don't have access, the synthetic data above demonstrates the same logic.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# UNCOMMENT THE BLOCK BELOW TO CONNECT TO NUVOLOS (requires credentials)
# ──────────────────────────────────────────────────────────────────────

# from pipeline.ingest import connect_nuvolos, get_transcript, get_transcripts
#
# # Connect (will try token -> RSA key -> browser SSO)
# con = connect_nuvolos()
#
# # Retrieve a single transcript
# ec = get_transcript("Apple Inc.", "2024", "3", con)
# print(f"Retrieved: {ec.company_name} {ec.year} Q{ec.quarter}")
# print(f"Transcript length: {len(ec.transcript):,} characters")
# print(f"Company ID: {ec.company_id}, Transcript ID: {ec.transcript_id}")
# print(f"\nFirst 500 chars:\n{ec.transcript[:500]}")
#
# # Retrieve multiple transcripts
# calls = [
#     ("Apple Inc.", "2024", "3"),
#     ("Microsoft Corporation", "2024", "2"),
#     ("Amazon.com, Inc.", "2024", "1"),
# ]
# transcripts = get_transcripts(calls, con)
# for t in transcripts:
#     print(f"  {t.company_name} {t.year} Q{t.quarter}: {len(t.transcript):,} chars")

print("(Uncomment the code above to test with real Nuvolos data)")

---

# Part 4: The Autoresearch Contract

The scoring and ingest stages feed into the **autoresearch loop** — an autonomous experiment system where an AI agent iterates on prediction strategies overnight.

## The three-file contract

| File | Who edits | Purpose |
|------|-----------|---------|
| `src/pipeline/evaluate.py` | **Nobody** (read-only) | Fixed evaluation harness. Runs the full pipeline, prints `METRIC: <float>` |
| `src/pipeline/experiment.py` | **Claude Code agent** | The sandbox. Feature engineering + scoring logic. Every experiment is a diff here. |
| `program.md` | **Human** | Research directions, constraints, stopping criteria |

## The experiment.py contract

The agent can only modify `experiment.py`, which must expose exactly two functions:

```python
def build_features(df: pl.DataFrame, settings: Settings) -> pl.DataFrame:
    """Add feature columns. Must not drop the target_variable column."""

def score(context: str, settings: Settings) -> float:
    """Score earnings context. Returns float in [-1.0, 1.0]."""
```

Let's verify the baseline implementations:

In [ ]:
import inspect
import polars as pl
from pipeline.experiment import build_features, score

# Verify signatures are intact
print("=== experiment.py signature verification ===\n")

bf_sig = inspect.signature(build_features)
print(f"build_features{bf_sig}")
print(f"  Parameters: {list(bf_sig.parameters.keys())}")

sc_sig = inspect.signature(score)
print(f"\nscore{sc_sig}")
print(f"  Parameters: {list(sc_sig.parameters.keys())}")

# Test baseline build_features (passthrough)
df = pl.DataFrame({
    "company": ["Acme", "Beta", "Gamma"],
    "oos_r_squared": [0.12, 0.08, 0.15],
    "revenue_growth": [0.05, -0.02, 0.10],
})
result_df = build_features(df, settings)

print(f"\nbuild_features baseline test:")
print(f"  Input:  {df.shape[0]} rows, {df.shape[1]} cols, columns={df.columns}")
print(f"  Output: {result_df.shape[0]} rows, {result_df.shape[1]} cols, columns={result_df.columns}")
assert "oos_r_squared" in result_df.columns, "Target column must be preserved!"
print(f"  OK: target column '{settings.target_variable}' preserved")

# Test baseline score (returns 0.0)
score_result = score("We expect revenue to grow 15% next quarter.", settings)
print(f"\nscore baseline test:")
print(f"  Input:  'We expect revenue to grow 15% next quarter.'")
print(f"  Output: {score_result}")
assert -1.0 <= score_result <= 1.0, "Score must be in [-1, 1]"
print(f"  OK: score {score_result} is in [-1.0, 1.0]")

## 4.1 Running the evaluator

The evaluation harness is the **fixed metric contract**. It always:
1. Loads settings from `global_params.yaml`
2. Runs the full pipeline (ingest -> features -> scoring -> modeling)
3. Prints exactly one line: `METRIC: <float>`
4. Exits 0 on success, non-zero on failure
5. Completes within 120 seconds

The current baseline returns `METRIC: 0.0` (placeholder — will be implemented in Stage 3).

In [ ]:
from pipeline.evaluate import main as evaluate_main
import io
import sys

# Capture stdout to read the METRIC line
old_stdout = sys.stdout
sys.stdout = buffer = io.StringIO()

evaluate_main()

sys.stdout = old_stdout
output = buffer.getvalue().strip()

print(f"Evaluator output: '{output}'")

# Parse the metric
assert output.startswith("METRIC:"), f"Expected 'METRIC: <float>', got '{output}'"
metric_value = float(output.split("METRIC:")[1].strip())
print(f"Parsed metric:   {metric_value}")
print(f"Type:            {type(metric_value).__name__}")
print(f"\n(Baseline = 0.0 — will improve as stages 3-5 are implemented)")

---

# Part 5: Summary

## What we tested

| Component | Module | Status |
|-----------|--------|--------|
| Global settings (YAML loading + validation) | `pipeline.settings` | Working |
| Transcript chunking | `pipeline.scoring.chunker` | Working |
| Entity neutering helpers | `pipeline.scoring.neutering` | Working |
| Pydantic schemas (all 5 stages) | `pipeline.scoring.schemas` | Working |
| Score computation (linear + log) | `pipeline.scoring.score_computer` | Working |
| Result building (EarningsCallResult) | `pipeline.scoring.score_computer` | Working |
| LLM client (mocked) | `pipeline.llm` | Working |
| Full LLM pipeline (mocked, 5 stages) | `pipeline.scoring.llm_pipeline` | Working |
| Loughran-McDonald dictionary scoring | `pipeline.scoring.lm_pipeline` | Working |
| SQL query generation | `pipeline.ingest.transcript` | Working |
| Transcript reconstruction | `pipeline.ingest.transcript` | Working |
| experiment.py signature contract | `pipeline.experiment` | Working |
| evaluate.py metric contract | `pipeline.evaluate` | Working |

## Data flow recap

```
[Nuvolos/Snowflake]                      [global_params.yaml]
       |                                         |
       v                                         v
  INGEST (transcript.py)                  SETTINGS (settings.py)
       |                                    |    |    |
       v                                    v    v    v
  Raw transcript text              LLM config  Eval  Features
       |                              |
       +--- LLM pipeline (5 stages) --+-- chunk_size, temperature, model
       |                              |
       +--- LM pipeline (dictionary) -+
       |
       v
  Structured scores (per category x sentiment)
       |
       v
  [experiment.py] build_features() + score()   <-- agent sandbox
       |
       v
  [evaluate.py] METRIC: <float>                <-- fixed harness
```

## Next steps

- **Stage 3**: Build the fixed evaluation harness (load data, run experiment, compute OOS R²)
- **Stage 4**: Baseline experiment.py (port feature engineering from notebooks)
- **Stage 5**: Populate program.md and run the first manual autoresearch loop